# File Sorting
This script is used for sort behavior recordings into different categories (Tmaze or Linear track)
1. The script can extract sample frames, enabling manually label for training ANN to identify videos and put them into different categories. When labeling frame, press q to exit viewing mode, enter LinearTrack, TMaze, or Other (for sleeping box) to manually identify frames. These date will be used to train ANN. Then ANN will be used to process all video files (containing 'behavior' in the file name). 
2. All videos will be categorised and saved in .csv files. 
3. Final processed files will be saved as {mouse_id}_{round_name}_FinalProcessed.csv.

### Import Libraries and Define Helper Functions

In [44]:
# Set mouse ID and round name dynamically (place this at the top of the notebook)
mouse_id = 'Mouse1639'
round_name = 'ThirdRound'

print(f"Mouse ID: {mouse_id}, Round: {round_name}")

Mouse ID: Mouse1639, Round: ThirdRound


In [22]:
import cv2
import os
import re
import csv
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
import numpy as np
from tqdm import tqdm  # For progress bar

# Set the fixed frame size for training and prediction (resize all frames to this size)
frame_size = (128, 128)  # Ensure this is consistent between training and prediction

# Function to extract and resize frame (grayscale)
def extract_and_resize_frame(video_path, fraction):
    video = cv2.VideoCapture(video_path)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idx = int(total_frames * fraction)
    video.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    success, frame = video.read()
    
    if success:
        # Convert to grayscale and resize the frame to the fixed size
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame_resized = cv2.resize(frame_gray, frame_size)
        return frame_resized
    else:
        print(f"Failed to extract frame from {video_path}")
        return None

# Function to save the extracted frame to the appropriate folder for labeling
def save_frame(frame, video_path, output_dir, label):
    output_file = os.path.join(output_dir, label, f'frame_{os.path.basename(video_path)}.png')
    cv2.imwrite(output_file, frame)
    print(f"Extracted frame saved to: {output_file}")

### Prepare Training Data from Labeled Frames - do not change the directory

In [23]:
# Initialize data and labels for training
data = []
labels = []

# Output_directory is for training data - do not change this directory 
# unless you are recreating a new dataset for training the model
output_directory = '/Volumes/Elements/Ruoxi/ThirdRound_ManuallyLabeledForTraining'

# Create the label folders (TMaze, LinearTrack, and Other)
Path(os.path.join(output_directory, 'TMaze')).mkdir(parents=True, exist_ok=True)
Path(os.path.join(output_directory, 'LinearTrack')).mkdir(parents=True, exist_ok=True)
Path(os.path.join(output_directory, 'Other')).mkdir(parents=True, exist_ok=True)

# Step 1: Load and process labeled data
for label in ['TMaze', 'LinearTrack', 'Other']:
    folder_path = os.path.join(output_directory, label)
    for frame_file in os.listdir(folder_path):
        frame_path = os.path.join(folder_path, frame_file)
        frame = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)  # Read as grayscale
        if frame is not None:
            # Resize the frame to the fixed size
            frame_resized = cv2.resize(frame, frame_size)
            data.append(frame_resized.flatten())  # Flatten the resized image
            labels.append(label)

# Convert data and labels to numpy arrays for training
data = np.array(data)
labels = np.array(labels)

# Step 2: Encoding the labels (TMaze -> 0, LinearTrack -> 1, Other -> 2)
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

In [17]:
# Initialize data and labels for training
data = []
labels = []

# Path to manually labeled data
output_directory = '/Volumes/Elements/Ruoxi/ThirdRound_ManuallyLabeledForTraining'

# List of label folders
label_folders = ['TMaze', 'LinearTrack', 'Other']

# Set the frame size for resizing
frame_size = (128, 128)  # Ensure this matches what you used during labeling

# Step 1: Load and process labeled data
for label in label_folders:
    folder_path = os.path.join(output_directory, label)
    
    # Loop through all the files in each folder
    for frame_file in os.listdir(folder_path):
        frame_path = os.path.join(folder_path, frame_file)
        
        # Read the image as grayscale
        frame = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
        
        if frame is not None:
            # Resize the frame to the fixed size used during training
            frame_resized = cv2.resize(frame, frame_size)
            
            # Flatten the resized image and append it to the data list
            data.append(frame_resized.flatten())
            
            # Append the label to the labels list
            labels.append(label)

# Convert data and labels to numpy arrays for training
data = np.array(data)
labels = np.array(labels)

# Step 2: Encode the labels (TMaze -> 0, LinearTrack -> 1, Other -> 2)
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

# Now you can use `data` and `encoded_labels` for training your model

### Train the Neural Network (MLPClassifier)

In [24]:
# Step 3: Train-Test Split for evaluation purposes
X_train, X_test, y_train, y_test = train_test_split(data, encoded_labels, test_size=0.2, random_state=42)

# Step 4: Define and Train the Neural Network (MLPClassifier)
clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, activation='relu', solver='adam', random_state=42)
clf.fit(X_train, y_train)

MLPClassifier(max_iter=500, random_state=42)

### Process Video Files and Predict Labels for New Frames

In [45]:
# Folder for videos that need to be labeled
# First round directory 
#video_directory = f'/Volumes/Elements/AGING MICE/{mouse_id}'

# 2, 3 round directory
video_directory = f'/Volumes/Elements/AGING MICE/{mouse_id}/{round_name}'

# Folder for saving the labeled frames
new_output_directory = f'/Volumes/Elements/Ruoxi/{mouse_id}/{round_name}/{round_name}_ANN_LabeledFrames'

# Ensure output directory and subfolders exist
Path(os.path.join(new_output_directory, 'TMaze')).mkdir(parents=True, exist_ok=True)
Path(os.path.join(new_output_directory, 'LinearTrack')).mkdir(parents=True, exist_ok=True)
Path(os.path.join(new_output_directory, 'Other')).mkdir(parents=True, exist_ok=True)

print(f"Video directory: {video_directory}")
print(f"Labeled frames will be saved to: {new_output_directory}")

Video directory: /Volumes/Elements/AGING MICE/Mouse1639/ThirdRound
Labeled frames will be saved to: /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ThirdRound_ANN_LabeledFrames


In [46]:
# Step 5: Use the trained model to categorize new frames from all video files

Path(os.path.join(new_output_directory, 'TMaze')).mkdir(parents=True, exist_ok=True)
Path(os.path.join(new_output_directory, 'LinearTrack')).mkdir(parents=True, exist_ok=True)
Path(os.path.join(new_output_directory, 'Other')).mkdir(parents=True, exist_ok=True)

# Get the list of video files to process that contain 'behavior' in their filename
video_files = [file for file in os.listdir(video_directory) if 'behavior' in file and file.endswith('.avi')]

# Progress bar to show processing progress
with tqdm(total=len(video_files), desc="Processing Videos", unit="video") as pbar:
    for video_file in video_files:
        video_path = os.path.join(video_directory, video_file)
        frame = extract_and_resize_frame(video_path, 2/3)  # Ensure frame is resized
        if frame is not None:
            frame_flatten = frame.flatten().reshape(1, -1)  # Flatten and reshape for prediction
            predicted_label = clf.predict(frame_flatten)
            label_name = label_encoder.inverse_transform(predicted_label)[0]
            
            # Save the predicted frame to the corresponding folder
            output_file = os.path.join(new_output_directory, label_name, f'frame_{os.path.basename(video_path)}.png')
            cv2.imwrite(output_file, frame)
            print(f"Predicted label for {video_file}: {label_name} - Saved to {output_file}")
        
        # Update the progress bar
        pbar.update(1)

Processing Videos:   1%|          | 1/128 [00:00<00:27,  4.69video/s]

Predicted label for behavior2024-10-02T14_09_12.avi: LinearTrack - Saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ThirdRound_ANN_LabeledFrames/LinearTrack/frame_behavior2024-10-02T14_09_12.avi.png


Processing Videos:   2%|▏         | 3/128 [00:00<00:23,  5.42video/s]

Predicted label for behavior2024-10-02T14_41_31.avi: TMaze - Saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ThirdRound_ANN_LabeledFrames/TMaze/frame_behavior2024-10-02T14_41_31.avi.png
Predicted label for behavior2024-10-03T13_13_31.avi: TMaze - Saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ThirdRound_ANN_LabeledFrames/TMaze/frame_behavior2024-10-03T13_13_31.avi.png


Processing Videos:   3%|▎         | 4/128 [00:00<00:21,  5.90video/s]

...

Predicted label for behaviorLinear2024-10-02T14_09_12.avi: Other - Saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ThirdRound_ANN_LabeledFrames/Other/frame_behaviorLinear2024-10-02T14_09_12.avi.png
Predicted label for behaviorLinear2024-10-02T14_41_31.avi: Other - Saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ThirdRound_ANN_LabeledFrames/Other/frame_behaviorLinear2024-10-02T14_41_31.avi.png


Processing Videos:  99%|█████████▉| 127/128 [00:22<00:00,  7.88video/s]

Predicted label for behaviorLinear2024-10-03T13_13_31.avi: Other - Saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ThirdRound_ANN_LabeledFrames/Other/frame_behaviorLinear2024-10-03T13_13_31.avi.png
Predicted label for behaviorLinear2024-10-03T13_14_17.avi: Other - Saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ThirdRound_ANN_LabeledFrames/Other/frame_behaviorLinear2024-10-03T13_14_17.avi.png
Predicted label for behaviorLinear2024-10-03T13_15_08.avi: Other - Saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ThirdRound_ANN_LabeledFrames/Other/frame_behaviorLinear2024-10-03T13_15_08.avi.png


Processing Videos: 100%|██████████| 128/128 [00:22<00:00,  5.66video/s]

Predicted label for behaviorLinear2024-10-03T13_46_50.avi: Other - Saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ThirdRound_ANN_LabeledFrames/Other/frame_behaviorLinear2024-10-03T13_46_50.avi.png


###  (Skip this)Check if All Files Have Been Processed and Show Progress

In [9]:
# Check if all video files have corresponding labeled frames
labeled_frame_files = []
for root, dirs, files in os.walk(new_output_directory):
    for file in files:
        if file.endswith('.png'):
            labeled_frame_files.append(os.path.basename(file).split('frame_')[-1].rsplit('.', 1)[0])

missing_files = [file for file in video_files if os.path.splitext(file)[0] not in labeled_frame_files]

# Output missing files, if any
if len(missing_files) == 0:
    print("All video files have corresponding labeled frames.")
else:
    print(f"Missing labeled frames for the following video files ({len(missing_files)} files):")
    for missing_file in missing_files:
        print(missing_file)

All video files have corresponding labeled frames.


In [10]:
def extract_and_resize_frame(video_path, fraction):
    video = cv2.VideoCapture(video_path)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idx = int(total_frames * fraction)
    video.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    success, frame = video.read()
    
    if success:
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame_resized = cv2.resize(frame_gray, frame_size)
        return frame_resized
    else:
        print(f"Failed to extract frame from {video_path}. Total frames: {total_frames}, Frame index: {frame_idx}")
        return None

In [11]:
def save_frame(frame, video_path, output_dir, label):
    try:
        output_file = os.path.join(output_dir, label, f'frame_{os.path.basename(video_path)}.png')
        cv2.imwrite(output_file, frame)
        if not os.path.exists(output_file):
            print(f"Failed to save frame to: {output_file}")
        else:
            print(f"Successfully saved frame to: {output_file}")
    except Exception as e:
        print(f"Error saving frame for {video_path}: {e}")

In [12]:
# Check if all video files have corresponding labeled frames
labeled_frame_files = []
for root, dirs, files in os.walk(new_output_directory):
    for file in files:
        if file.endswith('.png'):
            labeled_frame_files.append(os.path.basename(file).split('frame_')[-1].rsplit('.', 1)[0])

missing_files = [file for file in video_files if os.path.splitext(file)[0] not in labeled_frame_files]

# Output missing files, if any
if len(missing_files) == 0:
    print("All video files have corresponding labeled frames.")
else:
    print(f"Missing labeled frames for the following video files ({len(missing_files)} files):")
    for missing_file in missing_files:
        print(f"Video file: {missing_file}")
        corresponding_frame = f"frame_{os.path.basename(missing_file)}.png"
        print(f"Looking for corresponding frame: {corresponding_frame}")

All video files have corresponding labeled frames.


In [13]:
for missing_file in missing_files:
    video_path = os.path.join(video_directory, missing_file)
    frame = extract_and_resize_frame(video_path, 2/3)  # Ensure frame is resized
    if frame is not None:
        frame_flatten = frame.flatten().reshape(1, -1)  # Flatten and reshape for prediction
        predicted_label = clf.predict(frame_flatten)
        label_name = label_encoder.inverse_transform(predicted_label)[0]
        
        output_file = os.path.join(new_output_directory, label_name, f'frame_{os.path.basename(video_path)}.png')
        save_frame(frame, video_path, new_output_directory, label_name)

## (optional) Check if frames catch all video files
This script is for checking if the neural network captured frames from every video files.

In [47]:
# Define the paths to the ANN labeled frames and video files
ann_labeledframes_dir = new_output_directory  # Path to ANN_labeledframes folder

In [48]:
# Function to extract the filename without extensions
def extract_name_without_extension(filename):
    return os.path.splitext(filename)[0].strip()

# Get all .avi filenames from mouse1639 (only check filenames with 'behavior')
mouse1639_files = [f for f in os.listdir(video_directory) if 'behavior' in f]
mouse1639_filenames = [extract_name_without_extension(f) for f in mouse1639_files]

# Debugging: Print all mouse1639 filenames
print("Video directory filenames:")
for f in mouse1639_filenames:
    print(f)

# Get all frame files from ANN_labeledframes (recursively check subfolders)
ann_frame_files = []
for root, dirs, files in os.walk(ann_labeledframes_dir):
    for file in files:
        if file.startswith('frame_') and file.endswith('.avi.png'):
            ann_frame_files.append(file)

# Extract the base part of filenames from ANN frames and remove 'frame_' prefix
ann_filenames = [extract_name_without_extension(f).replace('frame_', '').replace('.avi', '') for f in ann_frame_files]

# Debugging: Print all ANN filenames for verification
print("\nANN labeled frame filenames (without 'frame_' prefix and .avi extension):")
for f in ann_filenames:
    print(f)

# Compare and find missing files in ANN_labeledframes
missing_files = [f for f in mouse1639_filenames if f not in ann_filenames]

# Output the missing files
if missing_files:
    print(f"\nMissing labeled frames for the following video files ({len(missing_files)}):")
    for file in missing_files:
        print(file)
else:
    print("All files in video directory have corresponding labeled frames in ANN_labeledframes.")

Video directory filenames:
behavior2024-10-02T14_09_12
behavior2024-10-02T14_41_31
behavior2024-10-03T13_13_31
behavior2024-10-03T13_14_17
behavior2024-10-03T13_15_08
behavior2024-10-03T13_46_50
behavior2024-10-04T12_17_06
behavior2024-10-04T12_50_19
behavior2024-10-05T15_31_25
behavior2024-10-05T16_06_36
behavior2024-10-06T13_14_52
behavior2024-10-06T13_17_01
behavior2024-10-06T13_27_29
behaviorLinear2024-10-04T12_50_19
behaviorLinear2024-10-05T15_31_25
behaviorLinear2024-10-05T16_06_36
behaviorLinear2024-10-06T13_14_52
behaviorLinear2024-10-06T13_17_01
behaviorLinear2024-10-06T13_27_29
behaviorLinear2024-10-09T12_44_33
behaviorLinear2024-10-09T13_15_21
behaviorLinear2024-10-10T13_24_35
behaviorLinear2024-10-10T13_55_26
behaviorLinear2024-10-11T13_49_03
behaviorLinear2024-10-11T14_21_13
behaviorLinear2024-10-12T15_23_44
behaviorLinear2024-10-12T15_53_22
behaviorLinear2024-10-12T16_28_01
behaviorLinear2024-10-13T12_44_11
behaviorLinear2024-10-13T12_45_10
behaviorLinear2024-10-13T12_46_

# Extract name and sort in csv file

In [49]:
# Paths to directories and output files
main_folder_path = f'/Volumes/Elements/Ruoxi/{mouse_id}/{round_name}/{round_name}_ANN_LabeledFrames'
extract_path = f'/Volumes/Elements/Ruoxi/{mouse_id}/{round_name}/ProcessedCSV/{round_name}_extracted.csv'

# Initialize a list to store extracted details
extracted_data = []

# Loop through each subfolder (Other, TMaze, LinearTrack)
for group_name in os.listdir(main_folder_path):
    group_folder_path = os.path.join(main_folder_path, group_name)
    
    if os.path.isdir(group_folder_path):  # Ensure it’s a folder
        # Loop through each file in the subfolder
        for file_name in os.listdir(group_folder_path):
            if file_name.startswith('frame_') and file_name.endswith('.png'):
                # Extract the original filename part after 'frame_' and before '.avi'
                original_file_name = file_name.replace('frame_', '').replace('.avi.png', '')
                
                # Use regex to extract the date and time (date before 'T' and time after 'T')
                match = re.search(r'(\d{4}-\d{2}-\d{2})T(\d{2}_\d{2}_\d{2})', original_file_name)
                
                if match:
                    date = match.group(1).replace('-', '')  # Format date as YYYYMMDD
                    time = match.group(2).replace('_', '')  # Format time as HHMMSS
                    
                    # Append the extracted data (original filename, date, time, group name) to the list
                    extracted_data.append({
                        'original_file_name': original_file_name,
                        'date': date,
                        'time': time,
                        'group_name': group_name  # Include the folder name (group)
                    })

# Output the extracted data to a CSV file
with open(extract_path, mode='w', newline='') as csv_file:
    fieldnames = ['original_file_name', 'date', 'time', 'group_name']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    
    # Write header
    writer.writeheader()
    
    # Write rows
    for data in extracted_data:
        writer.writerow(data)

print(f"Extracted data saved to {extract_path}")

Extracted data saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ProcessedCSV/ThirdRound_extracted.csv


## Sort files in csv using panda

In [50]:
import pandas as pd

# Path to the existing CSV file
extract_path = f'/Volumes/Elements/Ruoxi/{mouse_id}/{round_name}/ProcessedCSV/{round_name}_extracted.csv'
sort_path = f'/Volumes/Elements/Ruoxi/{mouse_id}/{round_name}/ProcessedCSV/{round_name}_sorted.csv'

# Load the CSV into a pandas DataFrame
df = pd.read_csv(extract_path)

# Combine the 'date' and 'time' columns into a single datetime column for sorting
df['datetime'] = pd.to_datetime(df['date'].astype(str) + df['time'].astype(str), format='%Y%m%d%H%M%S')

# Sort the DataFrame based on the combined datetime
df_sorted = df.sort_values(by='datetime', ascending=True)

# Output the sorted DataFrame to a new CSV file
df_sorted.to_csv(sort_path, index=False)

print(f"Sorted data saved to {sort_path}")


Sorted data saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ProcessedCSV/ThirdRound_sorted.csv


### Sort and saved file in csv as desired
Here we sort the file based on maze type and time

##### Add column specify: LinearTrack or TMaze, and first track of the day

In [51]:
updated_path = f'/Volumes/Elements/Ruoxi/{mouse_id}/{round_name}/ProcessedCSV/{round_name}_UpdatedCol.csv'

# Load the CSV into a pandas DataFrame
df = pd.read_csv(extract_path)

# Step 1: Add 'Linear' and 'TMaze' columns based on the 'group_name' column
df['Linear'] = df['group_name'].apply(lambda x: 1 if 'LinearTrack' in x else 0)
df['TMaze'] = df['group_name'].apply(lambda x: 1 if 'TMaze' in x else 0)

# Step 2: Initialize the "First_track_of_the_day" column to 0
df['First_track_of_the_day'] = 0

# Step 3: Iterate through each unique date, and mark the first occurrence of either LinearTrack or TMaze
for date in df['date'].unique():
    # Filter the DataFrame for the current date
    day_df = df[df['date'] == date]
    
    # Sort by time to find the first occurrence in the day
    day_df_sorted = day_df.sort_values(by='time', ascending=True)
    
    # Find the first occurrence of LinearTrack and TMaze
    first_linear = day_df_sorted[day_df_sorted['Linear'] == 1].head(1)  # First LinearTrack row
    first_tmaze = day_df_sorted[day_df_sorted['TMaze'] == 1].head(1)    # First TMaze row
    
    if not first_linear.empty and not first_tmaze.empty:
        # Compare the time values to see which comes first
        if first_linear['time'].values[0] < first_tmaze['time'].values[0]:
            df.loc[first_linear.index, 'First_track_of_the_day'] = 1  # LinearTrack is first
        else:
            df.loc[first_tmaze.index, 'First_track_of_the_day'] = 1  # TMaze is first
    elif not first_linear.empty:
        # Only LinearTrack found for the day
        df.loc[first_linear.index, 'First_track_of_the_day'] = 1
    elif not first_tmaze.empty:
        # Only TMaze found for the day
        df.loc[first_tmaze.index, 'First_track_of_the_day'] = 1

# Step 4: Sort the updated DataFrame based on the combined datetime (date + time)
df['datetime'] = pd.to_datetime(df['date'].astype(str) + df['time'].astype(str), format='%Y%m%d%H%M%S')
df_sorted = df.sort_values(by='datetime', ascending=True)

# Step 5: Output the sorted DataFrame to a new CSV file
df_sorted.to_csv(updated_path, index=False)

print(f"Updated and sorted data saved to {updated_path}")

Updated and sorted data saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ProcessedCSV/ThirdRound_UpdatedCol.csv


##### Add last column: time since last behavior

In [52]:
# Final processed data CSV file path
finalprocessed_path = f'/Volumes/Elements/Ruoxi/{mouse_id}/{round_name}/ProcessedCSV/{round_name}_FinalProcessed.csv'

# Step 4: Compute "Time since last behavior"
df_sorted['Time_since_last_behavior'] = pd.NaT  # Initialize column with NaT (not a time)

previous_other_time = None  # To track the datetime of the previous "Other" behavior

for date in df_sorted['date'].unique():
    # Filter the DataFrame for the current date
    day_df = df_sorted[df_sorted['date'] == date]
    
    # Find the first "Other" event of the day
    first_other = day_df[day_df['group_name'] == 'Other'].sort_values(by='datetime').head(1)
    
    if not first_other.empty:
        # If we found an "Other" event and there was a previous "Other" from the day before
        if previous_other_time is not None:
            time_difference = first_other['datetime'].values[0] - previous_other_time
            time_difference_hours = time_difference / np.timedelta64(1, 'h')  # Convert to hours
            
            # Set the computed time difference in the DataFrame
            df_sorted.loc[first_other.index, 'Time_since_last_behavior'] = time_difference_hours
        
        # Update previous_other_time to the current first "Other" datetime
        previous_other_time = first_other['datetime'].values[0]

# Extract the directory and original file name without extension
directory = os.path.dirname(finalprocessed_path)
original_file_name = os.path.basename(finalprocessed_path).replace('.csv', '')

# Create the new file name by appending the mouse ID
new_file_name = f"{mouse_id}_{original_file_name}.csv"

# Create the full path for the new file
new_csv_file_path = os.path.join(directory, new_file_name)

# Output the DataFrame to the newly named CSV file
df_sorted.to_csv(new_csv_file_path, index=False)

print(f"New data saved to {new_csv_file_path}")

New data saved to /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ProcessedCSV/Mouse1639_ThirdRound_FinalProcessed.csv


/var/folders/2n/r280w1zd71gfk6x3vq7hlqzw0000gn/T/ipykernel_62338/3253613772.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '23.071944444444444' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df_sorted.loc[first_other.index, 'Time_since_last_behavior'] = time_difference_hours


### Combine three round together
This step should be performed after all three round is processed

In [53]:
# File paths for the three rounds
first_round_file = f'/Volumes/Elements/Ruoxi/{mouse_id}/FirstRound/ProcessedCSV/{mouse_id}_FirstRound_FinalProcessed.csv'
second_round_file = f'/Volumes/Elements/Ruoxi/{mouse_id}/SecondRound/ProcessedCSV/{mouse_id}_SecondRound_FinalProcessed.csv'
third_round_file = f'/Volumes/Elements/Ruoxi/{mouse_id}/ThirdRound/ProcessedCSV/{mouse_id}_ThirdRound_FinalProcessed.csv'

# Initialize an empty list to hold the DataFrames
dfs = []

# Check if each file exists before loading and concatenating

if os.path.exists(first_round_file):
    print(f"Loading {first_round_file}")
    df_first = pd.read_csv(first_round_file)
    dfs.append(df_first)
else:
    print(f"File not found: {first_round_file}, skipping.")

if os.path.exists(second_round_file):
    print(f"Loading {second_round_file}")
    df_second = pd.read_csv(second_round_file)
    dfs.append(df_second)
else:
    print(f"File not found: {second_round_file}, skipping.")

if os.path.exists(third_round_file):
    print(f"Loading {third_round_file}")
    df_third = pd.read_csv(third_round_file)
    dfs.append(df_third)
else:
    print(f"File not found: {third_round_file}, skipping.")

# Concatenate the available DataFrames
if dfs:
    df_combined = pd.concat(dfs, ignore_index=True)

    # Output the combined DataFrame to a new CSV file
    combined_file_path = f'/Volumes/Elements/Ruoxi/{mouse_id}_CombinedRounds_FinalProcessed.csv'
    df_combined.to_csv(combined_file_path, index=False)

    print(f"Combined data saved to {combined_file_path}")
else:
    print("No files were loaded, skipping CSV creation.")

Loading /Volumes/Elements/Ruoxi/Mouse1639/FirstRound/ProcessedCSV/Mouse1639_FirstRound_FinalProcessed.csv
Loading /Volumes/Elements/Ruoxi/Mouse1639/SecondRound/ProcessedCSV/Mouse1639_SecondRound_FinalProcessed.csv
Loading /Volumes/Elements/Ruoxi/Mouse1639/ThirdRound/ProcessedCSV/Mouse1639_ThirdRound_FinalProcessed.csv
Combined data saved to /Volumes/Elements/Ruoxi/Mouse1639_CombinedRounds_FinalProcessed.csv


### Modify CSV files
This script is for adding new column corresponding to specific rounds, and information about sleepbox

In [54]:
# Load the CSV file (you can extend this to handle multiple rounds)
csv_file_path = combined_file_path
df = pd.read_csv(csv_file_path)

# Ensure that 'date' and 'time' columns are combined into a proper datetime format
df['datetime'] = pd.to_datetime(df['date'].astype(str) + df['time'].astype(str), format='%Y%m%d%H%M%S')

# Step 1: Create 'SleepBox1' and 'SleepBox2' columns
df['SleepBox1'] = 0
df['SleepBox2'] = 0

# Step 2: Assign SleepBox1 and SleepBox2 based on earliest and latest "Other" files of the day
for date in df['date'].unique():
    # Filter by date
    day_df = df[df['date'] == date]  
    other_files = day_df[day_df['group_name'] == 'Other']  # Filter "Other" group
    
    if not other_files.empty:
        # Assign SleepBox1 to the earliest "Other" file of the day
        earliest_other_index = other_files['datetime'].idxmin()
        df.loc[earliest_other_index, 'SleepBox1'] = 1
        
        # Assign SleepBox2 to the latest "Other" file of the day
        latest_other_index = other_files['datetime'].idxmax()
        df.loc[latest_other_index, 'SleepBox2'] = 1

# Step 3: Add 'RoundInfo' column and assign round information based on dates and group presence
df['RoundInfo'] = 'Unknown'

# Now we can access the month part of the datetime column
may_june_dates = (df['datetime'].dt.month.isin([5, 6]))
july_dates = (df['datetime'].dt.month == 7)
october_nov_dates = (df['datetime'].dt.month.isin([10, 11]))

# Step 4: Loop through each day and assign the round based on group presence
for date in df['date'].unique():
    day_df = df[df['date'] == date]
    has_lineartrack = (day_df['group_name'] == 'LinearTrack').any()
    has_tmaze = (day_df['group_name'] == 'TMaze').any()
    has_other = (day_df['group_name'] == 'Other').any()
    
    if may_june_dates[df['date'] == date].all():
        if has_lineartrack and has_other and not has_tmaze:
            # Round 1: May-June dates with LinearTrack and Other but no TMaze
            df.loc[df['date'] == date, 'RoundInfo'] = 'Round 1'
        elif has_lineartrack and has_tmaze and has_other:
            # Round 2: May-June dates with LinearTrack, TMaze, and Other
            df.loc[df['date'] == date, 'RoundInfo'] = 'Round 1.5'
    elif july_dates[df['date'] == date].all():
        df.loc[df['date'] == date, 'RoundInfo'] = 'Round 2'
    elif october_nov_dates[df['date'] == date].all():
        df.loc[df['date'] == date, 'RoundInfo'] = 'Round 3'

# Extract directory from original path
original_dir = os.path.dirname(csv_file_path)

# Extract the base file name without extension
base_file_name = os.path.basename(csv_file_path).replace('.csv', '')

# Create new file name with updated portion
new_file_name = f"{base_file_name}_Final.csv"

# Combine original directory with the new file name to create new path
new_csv_file_path = os.path.join(original_dir, new_file_name)

# Save the updated DataFrame to the new CSV file
df.to_csv(new_csv_file_path, index=False)

print(f"Updated data saved to: {new_csv_file_path}")

Updated data saved to: /Volumes/Elements/Ruoxi/Mouse1639_CombinedRounds_FinalProcessed_Final.csv


### Add duration column

In [55]:
# Set mouse ID dynamically
mouse_id = 'Mouse1639'  # Change this accordingly

# Function to get the duration of a video file in minutes:seconds format
def get_video_duration(video_path):
    video = cv2.VideoCapture(video_path)
    
    if not video.isOpened():
        print(f"Error opening video file: {video_path}")
        return None

    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = video.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        print(f"Error retrieving FPS for video: {video_path}")
        return None

    # Calculate duration in seconds
    duration_seconds = total_frames / fps
    
    # Convert seconds to minutes:seconds format
    minutes = int(duration_seconds // 60)
    seconds = int(duration_seconds % 60)
    
    return f"{minutes}:{seconds:02d}"

# Function to process video files and update the DataFrame with durations for a specific round
def add_video_durations_for_round(df, mouse_id, round_number, round_info_value):
    round_directory = f'/Volumes/Elements/AGING MICE/{mouse_id}/{round_number}'
    
    # Filter the DataFrame to include only the relevant rows for the current round
    relevant_rows = df[df['RoundInfo'] == round_info_value]
    
    for index, row in relevant_rows.iterrows():
        video_name = row['original_file_name']
        video_path = os.path.join(round_directory, f"{video_name}.avi")
        
        # Check if the video exists and if the 'duration_min' is NaN (empty)
        if pd.isna(row['duration_min']) and os.path.exists(video_path):
            print(f"Processing video: {video_path}")  # Debugging output
            duration_min_sec = get_video_duration(video_path)
            if duration_min_sec is not None:
                # Update the DataFrame with the video duration in MM:SS format
                df.at[index, 'duration_min'] = duration_min_sec
        elif not os.path.exists(video_path):
            print(f"Video not found: {video_path}")  # Debugging output

    return df


# Load the existing CSV file with all rounds combined
csv_file_path = f'/Volumes/Elements/Ruoxi/{mouse_id}_CombinedRounds_FinalProcessed_Final.csv'
df_csv = pd.read_csv(csv_file_path)

# Ensure the 'duration_min' column exists, otherwise initialize it
if 'duration_min' not in df_csv.columns:
    df_csv['duration_min'] = pd.NA

# Process First Round files (Round 1 and Round 1.5) and update the DataFrame
df_csv = add_video_durations_for_round(df_csv, mouse_id, 'FirstRound', 'Round 1')
df_csv = add_video_durations_for_round(df_csv, mouse_id, 'FirstRound', 'Round 1.5')

# Process Second Round files (Round 2) and update the DataFrame
df_csv = add_video_durations_for_round(df_csv, mouse_id, 'SecondRound', 'Round 2')

# Process Third Round files (Round 3) and update the DataFrame
df_csv = add_video_durations_for_round(df_csv, mouse_id, 'ThirdRound', 'Round 3')

# Save the updated DataFrame to a new CSV file
new_csv_file_path = f'/Volumes/Elements/Ruoxi/{mouse_id}_BehaviorFileAnalysis_Final.csv'
df_csv.to_csv(new_csv_file_path, index=False)

print(f"Updated CSV with video durations (MM:SS format) saved to {new_csv_file_path}")

Processing video: /Volumes/Elements/AGING MICE/Mouse1639/FirstRound/behavior2024-05-22T18_43_07.avi
Processing video: /Volumes/Elements/AGING MICE/Mouse1639/FirstRound/behaviorLinear2024-05-22T18_43_07.avi
Processing video: /Volumes/Elements/AGING MICE/Mouse1639/FirstRound/behavior2024-05-22T19_08_07.avi
Processing video: /Volumes/Elements/AGING MICE/Mouse1639/FirstRound/behaviorLinear2024-05-22T19_08_07.avi
Processing video: /Volumes/Elements/AGING MICE/Mouse1639/FirstRound/behavior2024-05-23T16_39_14.avi
Processing video: /Volumes/Elements/AGING MICE/Mouse1639/FirstRound/behaviorLinear2024-05-23T16_39_14.avi
Processing video: /Volumes/Elements/AGING MICE/Mouse1639/FirstRound/behaviorLinear2024-05-23T17_10_09.avi
Processing video: /Volumes/Elements/AGING MICE/Mouse1639/FirstRound/behavior2024-05-23T17_10_09.avi
Processing video: /Volumes/Elements/AGING MICE/Mouse1639/FirstRound/behaviorLinear2024-05-24T17_02_07.avi
Processing video: /Volumes/Elements/AGING MICE/Mouse1639/FirstRound/be